In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("marketing_team").getOrCreate()


dbutils.widgets.text("customer_path","")
dbutils.widgets.text("order_path","")

dbutils.widgets.text("silver_customer_path","")
dbutils.widgets.text("silver_order_path","")

customer_path = dbutils.widgets.get("customer_path")
order_path = dbutils.widgets.get("order_path")
silver_customer_path = dbutils.widgets.get("silver_customer_path")
silver_order_path = dbutils.widgets.get("silver_order_path")

#read csv files
df_customer = spark.read.csv(customer_path, header=True, inferSchema=True)
df_order = spark.read.csv(order_path, header=True, inferSchema=True)

# clean null values
df_customer=df_customer.filter(df_customer.customer_id.isNotNull()) 
df_order=df_order.filter(df_order.order_id.isNotNull() & df_order.customer_id.isNotNull())


#remove duplicate records
df_customer=df_customer.dropDuplicates(["customer_id"])
df_order=df_order.dropDuplicates(["order_id"])

print("loading into delta tables")

#load customer and order data into delta lake
df_customer.write.format("delta").mode("overwrite").save(silver_customer_path)
df_order.write.format("delta").mode("overwrite").save(silver_order_path)



    